<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# pdfs -> neo4j using SimpleKGPipeline
%pip install -U neo4j-graphrag neo4j pypdf python-dotenv openai --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 4.0 MB/s eta 0:00:00


In [ ]:
import os
import asyncio
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings.openai import BaseOpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
from google.colab import userdata

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
os.environ["NEO4J_URI"] = userdata.get("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = userdata.get("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = userdata.get("NEO4J_PASSWORD")
os.environ["NEO4J_DATABASE"] = userdata.get("NEO4J_DATABASE")

NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

In [ ]:
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
neo4j_driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


In [ ]:
from pypdf import PdfReader
PDF_PATH = Path("/content/Leave_Policy.pdf")

reader = PdfReader(PDF_PATH)
print(f"File: {PDF_PATH.name}")
print(f"Number of pages: {len(reader.pages)}")

File: Leave_Policy.pdf
Number of pages: 7


In [ ]:
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

class NemotronEmbeddings(BaseOpenAIEmbeddings):
    """
    NVIDIA Nemotron embeddings via NIM's OpenAI-compatible /v1/embeddings endpoint.
    Nemotron-family embedding models require an `input_type` of "passage" (indexing)
    or "query" (retrieval) on every request, so we inject it automatically here.
    """
    def __init__(self, model="nvidia/nemotron-3-embed-1b", input_type="passage", **kwargs):
        self.input_type = input_type
        super().__init__(model=model, **kwargs)

    def _initialize_client(self, **kwargs):
        return self.openai.OpenAI(**kwargs)

    def embed_query(self, text, **kwargs):
        kwargs.setdefault("extra_body", {"input_type": self.input_type})
        return super().embed_query(text, **kwargs)


llm = OpenAILLM(
    model_name="nvidia/nemotron-3-super-120b-a12b",
    model_params={
        "response_format": {"type": "json_object"},
    },
    base_url=NVIDIA_BASE_URL,
    api_key=os.getenv("NVIDIA_API_KEY"),
)

embedder = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    base_url=NVIDIA_BASE_URL,
    api_key=os.getenv("NVIDIA_API_KEY"),
    input_type="passage",
)

print("LLM and embedder configured (NVIDIA NIM - free tier)")

LLM and embedder configured (NVIDIA NIM - free tier)


In [ ]:
kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=neo4j_driver,
    embedder=embedder,
    neo4j_database=NEO4J_DATABASE,
    schema="FREE",
    from_file=True,
    perform_entity_resolution=True,
    on_error="IGNORE",
)
print("KG Pipeline Configured!")

KG Pipeline Configured!


In [ ]:
result = await kg_builder.run_async(file_path=str(PDF_PATH))  # await kg_builder.run_async(text="my text")  # if using from_file=False
print("Pipeline finished")
print(result)

Pipeline finished
run_id='70d67b06-ab55-4c00-98e9-0728335ad5e2' result={'resolver': {'number_of_nodes_to_resolve': 38, 'number_of_created_nodes': 24}}
